# 🐘 PostgreSQL Features Showcase with Calliope AI

Explore PostgreSQL-specific features and advanced capabilities using Calliope AI with real sample databases.

## 🗄️ PostgreSQL Databases Available

This notebook demonstrates features across **2 PostgreSQL databases**:
- **chinook** - Digital music store
- **northwind** - Business operations

## 🎯 What We'll Cover

1. PostgreSQL-specific data types (arrays, JSON, HSTORE)
2. Advanced string functions and regex
3. Window functions and analytics
4. Common Table Expressions (CTEs)
5. Array operations
6. JSON/JSONB operations
7. Full-text search
8. Advanced aggregates
9. Performance optimization with EXPLAIN ANALYZE

## ⚙️ Configuration

**Run this cell first!** Sets up the Calliope API endpoint.

In [ ]:
# =============================================================================
# AUTO-INSTALL DEPENDENCIES
# Run this cell to scan the notebook and install required packages
# =============================================================================
%%calliope ask claude --format code
Scan this notebook for all Python imports and library dependencies.
Generate a shell script that:
1. Checks if each required package is installed
2. Installs any missing packages using pip
Output only the executable shell commands, no explanations.


In [ ]:
import os

# Configure Calliope Data Agent API endpoint
API_BASE_URL = 'http://localhost:5000'  # Local development
# API_BASE_URL = 'http://data-agent:5000'  # Docker/JupyterHub

os.environ['CALLIOPE_API_BASE_URL'] = API_BASE_URL
print(f"✅ Calliope API configured: {API_BASE_URL}")

## ✅ Verify PostgreSQL Datasources

In [ ]:
import requests

response = requests.get('http://localhost:5000/api/datasources')
datasources = response.json()['datasources']

postgres_datasources = [ds for ds in datasources if ds['dialect'] == 'postgres']
print("✅ PostgreSQL Datasources Available:")
for ds in postgres_datasources:
    print(f"  • {ds['id']}: {ds['name']}")

## 🔤 Advanced String Functions & Regex

PostgreSQL has powerful string manipulation and pattern matching.

### String Concatenation and Formatting

In [ ]:
%%calliope run-sql chinook
SELECT 
    customer_id,
    first_name || ' ' || last_name as full_name,
    CONCAT(first_name, ' ', last_name) as full_name_concat,
    FORMAT('%s from %s, %s', company, city, country) as formatted_info,
    UPPER(LEFT(first_name, 1)) || UPPER(LEFT(last_name, 1)) as initials,
    LENGTH(first_name || last_name) as name_length
FROM customer
LIMIT 10

### Regular Expressions

In [ ]:
%%calliope run-sql chinook
SELECT 
    name,
    REGEXP_MATCH(name, '[0-9]+') as numbers_found,
    REGEXP_REPLACE(name, '[0-9]+', 'X') as numbers_replaced,
    name ~ '^[A-Z]' as starts_with_capital,
    name ~* 'rock' as contains_rock_case_insensitive
FROM track
WHERE name ~ '[0-9]+'
LIMIT 20

## 🪟 Advanced Window Functions

PostgreSQL has extensive window function support.

### Ranking and Distribution

In [ ]:
%%calliope run-sql northwind
SELECT 
    product_name,
    category_id,
    unit_price,
    RANK() OVER (PARTITION BY category_id ORDER BY unit_price DESC) as price_rank,
    DENSE_RANK() OVER (PARTITION BY category_id ORDER BY unit_price DESC) as price_dense_rank,
    PERCENT_RANK() OVER (PARTITION BY category_id ORDER BY unit_price) as price_percentile,
    NTILE(4) OVER (PARTITION BY category_id ORDER BY unit_price) as price_quartile,
    ROUND(CUME_DIST() OVER (PARTITION BY category_id ORDER BY unit_price)::numeric, 3) as cumulative_dist
FROM products
WHERE discontinued = 0
LIMIT 30

### LAG, LEAD, and Frame Clauses

In [ ]:
%%calliope run-sql chinook
SELECT 
    invoice_date::date,
    ROUND(total::numeric, 2) as daily_total,
    ROUND(LAG(total, 1) OVER (ORDER BY invoice_date)::numeric, 2) as previous_day,
    ROUND(LEAD(total, 1) OVER (ORDER BY invoice_date)::numeric, 2) as next_day,
    ROUND(AVG(total) OVER (
        ORDER BY invoice_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    )::numeric, 2) as moving_avg_7day,
    ROUND(SUM(total) OVER (
        ORDER BY invoice_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )::numeric, 2) as running_total
FROM invoice
ORDER BY invoice_date
LIMIT 30

## 🔗 Common Table Expressions (CTEs)

PostgreSQL has excellent CTE support including recursive queries.

### Multiple CTEs

In [ ]:
%%calliope run-sql northwind
WITH customer_orders AS (
    SELECT 
        customer_id,
        COUNT(*) as order_count,
        SUM(od.unit_price * od.quantity * (1 - od.discount)) as total_revenue
    FROM orders o
    JOIN order_details od ON o.order_id = od.order_id
    GROUP BY customer_id
),
customer_segments AS (
    SELECT 
        *,
        CASE 
            WHEN total_revenue > 10000 THEN 'VIP'
            WHEN total_revenue > 5000 THEN 'Gold'
            WHEN total_revenue > 1000 THEN 'Silver'
            ELSE 'Bronze'
        END as segment
    FROM customer_orders
)
SELECT 
    c.company_name,
    c.country,
    cs.order_count,
    ROUND(cs.total_revenue::numeric, 2) as total_revenue,
    cs.segment
FROM customers c
JOIN customer_segments cs ON c.customer_id = cs.customer_id
ORDER BY cs.total_revenue DESC
LIMIT 20

### Recursive CTEs

In [ ]:
%%calliope run-sql northwind
-- Generate Fibonacci sequence
WITH RECURSIVE fibonacci(n, fib_n, fib_n_plus_1) AS (
    SELECT 1, 0::bigint, 1::bigint
    UNION ALL
    SELECT n + 1, fib_n_plus_1, fib_n + fib_n_plus_1
    FROM fibonacci
    WHERE n < 20
)
SELECT n, fib_n as fibonacci_number
FROM fibonacci

## 📊 Advanced Aggregates

PostgreSQL offers statistical and advanced aggregate functions.

In [ ]:
%%calliope run-sql chinook
SELECT 
    g.name as genre,
    COUNT(*) as track_count,
    ROUND(AVG(t.milliseconds / 1000.0)::numeric, 2) as avg_duration_seconds,
    ROUND(STDDEV(t.milliseconds / 1000.0)::numeric, 2) as stddev_duration,
    ROUND(VARIANCE(t.milliseconds / 1000.0)::numeric, 2) as variance_duration,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY t.milliseconds / 1000.0) as median_duration,
    MODE() WITHIN GROUP (ORDER BY t.unit_price) as most_common_price,
    STRING_AGG(DISTINCT t.composer, ', ' ORDER BY t.composer) FILTER (WHERE t.composer IS NOT NULL) as composers
FROM genre g
JOIN track t ON g.genre_id = t.genre_id
GROUP BY g.genre_id, g.name
ORDER BY track_count DESC
LIMIT 15

## 📚 Array Operations

PostgreSQL's array support is unique among SQL databases.

In [ ]:
%%calliope run-sql northwind
-- Create and manipulate arrays
SELECT 
    customer_id,
    ARRAY_AGG(order_id ORDER BY order_date) as order_ids,
    ARRAY_AGG(order_date ORDER BY order_date) as order_dates,
    ARRAY_LENGTH(ARRAY_AGG(order_id), 1) as num_orders,
    (ARRAY_AGG(order_date ORDER BY order_date DESC))[1] as most_recent_order
FROM orders
GROUP BY customer_id
HAVING COUNT(*) >= 5
LIMIT 20

## 📅 Date/Time Functions

PostgreSQL has rich date/time manipulation capabilities.

In [ ]:
%%calliope run-sql chinook
SELECT 
    invoice_date,
    EXTRACT(YEAR FROM invoice_date) as year,
    EXTRACT(QUARTER FROM invoice_date) as quarter,
    EXTRACT(MONTH FROM invoice_date) as month,
    TO_CHAR(invoice_date, 'Day') as day_of_week,
    TO_CHAR(invoice_date, 'Month DD, YYYY') as formatted_date,
    AGE(CURRENT_DATE, invoice_date::date) as time_since_invoice,
    DATE_TRUNC('month', invoice_date) as month_start,
    invoice_date + INTERVAL '30 days' as due_date
FROM invoice
ORDER BY invoice_date DESC
LIMIT 15

## 🔢 Generate Series

Create sequences of numbers, dates, or timestamps.

In [ ]:
%%calliope run-sql northwind
-- Generate a date series and join with orders
WITH date_series AS (
    SELECT generate_series(
        '2023-01-01'::date,
        '2023-12-31'::date,
        '1 day'::interval
    )::date as date
)
SELECT 
    ds.date,
    TO_CHAR(ds.date, 'Day') as day_of_week,
    COUNT(o.order_id) as order_count,
    COALESCE(SUM(od.unit_price * od.quantity), 0) as daily_revenue
FROM date_series ds
LEFT JOIN orders o ON o.order_date::date = ds.date
LEFT JOIN order_details od ON o.order_id = od.order_id
GROUP BY ds.date
ORDER BY ds.date
LIMIT 30

## 🔀 LATERAL Joins

Powerful feature for correlated subqueries.

In [ ]:
%%calliope run-sql chinook
-- Get top 3 tracks for each genre
SELECT 
    g.name as genre,
    top_tracks.name as track_name,
    top_tracks.composer,
    ROUND((top_tracks.milliseconds / 60000.0)::numeric, 2) as duration_minutes
FROM genre g
CROSS JOIN LATERAL (
    SELECT t.name, t.composer, t.milliseconds
    FROM track t
    WHERE t.genre_id = g.genre_id
    ORDER BY t.milliseconds DESC
    LIMIT 3
) top_tracks
ORDER BY g.name, top_tracks.milliseconds DESC
LIMIT 30

## 🎯 FILTER Clause in Aggregates

In [ ]:
%%calliope run-sql northwind
SELECT 
    c.category_name,
    COUNT(*) as total_products,
    COUNT(*) FILTER (WHERE p.discontinued = 0) as active_products,
    COUNT(*) FILTER (WHERE p.discontinued = 1) as discontinued_products,
    ROUND(AVG(p.unit_price) FILTER (WHERE p.discontinued = 0)::numeric, 2) as avg_active_price,
    COUNT(*) FILTER (WHERE p.units_in_stock = 0) as out_of_stock,
    SUM(p.units_in_stock) FILTER (WHERE p.discontinued = 0) as total_stock
FROM categories c
JOIN products p ON c.category_id = p.category_id
GROUP BY c.category_id, c.category_name
ORDER BY total_products DESC

## 📊 GROUPING SETS, CUBE, and ROLLUP

In [ ]:
%%calliope run-sql chinook
SELECT 
    COALESCE(c.country, 'ALL COUNTRIES') as country,
    COALESCE(g.name, 'ALL GENRES') as genre,
    COUNT(DISTINCT i.invoice_id) as num_purchases,
    ROUND(SUM(il.unit_price * il.quantity)::numeric, 2) as total_revenue
FROM customer c
JOIN invoice i ON c.customer_id = i.customer_id
JOIN invoice_line il ON i.invoice_id = il.invoice_id
JOIN track t ON il.track_id = t.track_id
JOIN genre g ON t.genre_id = g.genre_id
GROUP BY GROUPING SETS (
    (c.country, g.name),
    (c.country),
    (g.name),
    ()
)
ORDER BY country NULLS LAST, genre NULLS LAST
LIMIT 50

## 🎯 DISTINCT ON

In [ ]:
%%calliope run-sql northwind
-- Get the most recent order for each customer
SELECT DISTINCT ON (customer_id)
    customer_id,
    order_id,
    order_date,
    shipped_date,
    freight
FROM orders
ORDER BY customer_id, order_date DESC
LIMIT 20

## ⚡ Performance Analysis with EXPLAIN ANALYZE

In [ ]:
%%calliope run-sql chinook
EXPLAIN ANALYZE
SELECT 
    c.country,
    COUNT(*) as customer_count,
    SUM(i.total) as total_revenue
FROM customer c
JOIN invoice i ON c.customer_id = i.customer_id
GROUP BY c.country
HAVING SUM(i.total) > 100
ORDER BY total_revenue DESC

## 🤖 Using Calliope AI for PostgreSQL-Specific Queries

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic
Using window functions and FILTER clauses, analyze purchase patterns by country and identify the top-performing genres in each country

In [ ]:
%%calliope ask-sql northwind --sql-model anthropic
Create a recursive CTE that shows the cumulative revenue growth month over month

In [ ]:
%%calliope ask-sql chinook --sql-model anthropic --to-ai --model claude3
Use LATERAL joins to find customers and their purchase history. Analyze spending patterns and provide recommendations.

## 📊 Summary

This notebook demonstrated PostgreSQL-specific features:

✅ **Advanced String Functions**: Regex, pattern matching  
✅ **Window Functions**: Comprehensive analytical capabilities  
✅ **CTEs**: Recursive queries, multiple CTEs  
✅ **Array Operations**: ARRAY_AGG, array manipulation  
✅ **Advanced Aggregates**: Statistical functions, FILTER clause  
✅ **Date/Time**: Rich manipulation and formatting  
✅ **Generate Series**: Create sequences  
✅ **LATERAL Joins**: Correlated subqueries  
✅ **GROUPING SETS**: Multi-level aggregations  
✅ **DISTINCT ON**: Unique PostgreSQL feature  
✅ **Performance**: EXPLAIN ANALYZE  

## 🎯 Next Steps

- Compare with MySQL features in `MySQL-Features-Showcase.ipynb`
- Explore individual datasources: Chinook, Northwind
- Build advanced analytical queries with Calliope AI
- Experiment with JSON/JSONB operations (if available in your schema)